# nb10.4 - Oster (2019) Delta and Bounds: How Much Selection on Unobservables Would Kill This Result?

*Week 10, Chapter 10.1 companion. Devon's crypto paper claims a positive effect of a new exchange's
launch on local DeFi adoption. The discussant says: "Your controls look fine, but your treatment is
non-random. What if there's an unobserved tech-savviness variable? Show me how much of it I would need
to drive your effect to zero." That sentence is exactly Oster's $\delta$.*

The reveal-the-trick lesson: **Oster's $\delta$ asks how big the unobservables would have to be -
relative to the observables - to explain away your estimated treatment effect.** A $\delta = 1$ means
"unobservables would have to be exactly as influential as the observables, jointly." Most published
papers using Oster's method report $\delta > 1$ as the bar for credibility. The harder companion
question is what value of $R^2_{\max}$ - the $R^2$ you would get if you could observe everything -
to plug in. Oster's empirical default is $R^2_{\max} = 1.3 \cdot \widetilde R^2$ where $\widetilde R^2$
is the $R^2$ from the fully-controlled regression. This notebook builds the math, recovers $\delta$ on
a DGP where we know the truth, and sweeps $R^2_{\max}$ to show how the answer moves.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

rng = np.random.default_rng(42)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

## 1. A DGP with observables, unobservables, and a known truth

We construct an outcome with three contributing pieces: a treatment $D$, an *observed* covariate index
$W_1$, and an *unobserved* covariate index $W_2$. The two indices are jointly distributed with the
treatment in a way that mimics selection: tech-savvy regions (high $W_1$ and high $W_2$) are more
likely to receive the treatment. We control the strength of selection so we can later ask "by how
much did selection on observables foreshadow selection on unobservables?"

Concretely, draw $(W_1, W_2, U) \sim \mathcal{N}(\mathbf{0}, I)$, and let
$$D = \mathbb{1}\{\eta + \lambda_1 W_1 + \lambda_2 W_2 + \nu > 0\},$$
$$Y = \tau \cdot D + \theta_1 W_1 + \theta_2 W_2 + \varepsilon.$$
We **observe** $D$ and $W_1$; we do **not** observe $W_2$. The truth is $\tau = 0.30$. The observables
explain part of the selection ($\lambda_1$), the unobservables explain the rest ($\lambda_2$). Oster's
question is: from the observed shift $(\widehat\tau_{\text{short}}, \widehat\tau_{\text{long}})$ and
the observed $R^2$ shift $(R^2_{\text{short}}, R^2_{\text{long}})$, can we extrapolate the bias caused
by $W_2$?

In [ ]:
def make_oster_dgp(rng, N=4000, tau=0.30,
                   theta1=0.6, theta2=0.6,
                   lambda1=0.5, lambda2=0.5):
    W1 = rng.normal(0, 1, size=N)
    W2 = rng.normal(0, 1, size=N)   # unobserved
    nu = rng.normal(0, 1, size=N)
    eps = rng.normal(0, 1, size=N)
    eta = -0.1
    D_index = eta + lambda1 * W1 + lambda2 * W2 + nu
    D = (D_index > 0).astype(int)
    Y = tau * D + theta1 * W1 + theta2 * W2 + eps
    df = pd.DataFrame({"D": D, "W1": W1, "W2": W2, "Y": Y})
    df.attrs["tau_true"] = tau
    return df

df = make_oster_dgp(rng)
print(f"true tau = {df.attrs['tau_true']}")
print(f"treatment share = {df['D'].mean():0.3f}")
df.head()

## 2. The short and long regressions

The "short" regression uses only the treatment; the "long" uses the treatment plus the *observed*
controls. We never see $W_2$.

In [ ]:
def run_short_long(df):
    Y = df["Y"].values
    D = df["D"].values
    W1 = df["W1"].values
    # Short: Y ~ const + D
    short = sm.OLS(Y, sm.add_constant(D)).fit()
    tau_short = short.params[1]; R2_short = short.rsquared
    # Long: Y ~ const + D + W1
    long_ = sm.OLS(Y, sm.add_constant(np.column_stack([D, W1]))).fit()
    tau_long = long_.params[1]; R2_long = long_.rsquared
    return tau_short, R2_short, tau_long, R2_long

tau_short, R2_short, tau_long, R2_long = run_short_long(df)
print(f"tau_short = {tau_short:0.4f}, R2_short = {R2_short:0.4f}")
print(f"tau_long  = {tau_long:0.4f}, R2_long  = {R2_long:0.4f}")
print(f"true tau  = {df.attrs['tau_true']}")

**Read the numbers.** The short estimate is biased *upward* (treatment is positively
correlated with both $W_1$ and $W_2$, both of which have positive effects on $Y$). Adding $W_1$ to the
regression brings $\widehat\tau$ closer to the truth but does not get all the way there - the
remaining gap is exactly what $W_2$ would close if we could observe it.

## 3. Oster's $\delta$ formula

Oster (2019, *Journal of Business & Economic Statistics* 37(2):187-204) defines $\delta$ as the
**ratio** of selection on unobservables to selection on observables (in coefficient-of-proportional-
selection terms). Under her main equation,
$$\beta^*(R^2_{\max}, \delta) \approx \widetilde\beta - \delta \cdot
   (\dot\beta - \widetilde\beta) \cdot \frac{R^2_{\max} - \widetilde R^2}{\widetilde R^2 - \dot R^2},$$
where $\widetilde\beta = \widehat\tau_{\text{long}}$, $\dot\beta = \widehat\tau_{\text{short}}$,
$\widetilde R^2 = R^2_{\text{long}}$, $\dot R^2 = R^2_{\text{short}}$, and $\beta^*$ is the
bias-adjusted treatment effect. Solving for $\delta^*$ that yields $\beta^* = 0$:
$$\delta^* = \frac{\widetilde\beta \cdot (\widetilde R^2 - \dot R^2)}
                  {(\dot\beta - \widetilde\beta) \cdot (R^2_{\max} - \widetilde R^2)}.$$
The convention is to compute $\delta^*$ for several plausible values of $R^2_{\max}$ and report the
range. The further $\delta^*$ is above 1, the harder it is to write off your effect as selection on
unobservables.

In [ ]:
def oster_delta(tau_short, R2_short, tau_long, R2_long, R2_max):
    # delta that drives beta_star to zero.
    num = tau_long * (R2_long - R2_short)
    den = (tau_short - tau_long) * (R2_max - R2_long)
    if abs(den) < 1e-10:
        return float("nan")
    return num / den

def oster_beta_star(tau_short, R2_short, tau_long, R2_long, R2_max, delta):
    return tau_long - delta * (tau_short - tau_long) * (R2_max - R2_long) / (R2_long - R2_short)

# Oster's recommended default: R2_max = min(1.3 * R2_long, 1)
R2_max_default = min(1.3 * R2_long, 0.99)
delta_star = oster_delta(tau_short, R2_short, tau_long, R2_long, R2_max_default)
print(f"R2_max (default 1.3 x R2_long) = {R2_max_default:0.3f}")
print(f"delta* (drives beta to 0)      = {delta_star:0.3f}")
print(f"beta_star at delta=1, R2_max=default: "
      f"{oster_beta_star(tau_short, R2_short, tau_long, R2_long, R2_max_default, 1.0):0.3f}")

**Read the $\delta^*$ number.** A value $> 1$ means that to wipe out the estimated effect, the
unobservables would have to be *more* influential than the observed controls. In Oster's calibration
studies, randomized-experiment papers (where the truth is known to be zero bias) have $\delta^* > 5$;
observational papers in top journals tend to cluster around $\delta^* \approx 1-2$. A paper that
reports $\delta^* < 1$ should be much more cautious about its causal claim.

## 4. Sweep $R^2_{\max}$: how brittle is $\delta^*$?

The $R^2_{\max}$ assumption is genuinely contestable. The default $1.3 \cdot \widetilde R^2$ is an
empirical regularity from Oster (2019). Some referees insist on $R^2_{\max} = 1$ (the unobservables
explain *everything* left over). Some accept a tighter cap. We sweep.

In [ ]:
rmax_grid = np.linspace(R2_long + 0.001, 0.999, 80)
deltas = [oster_delta(tau_short, R2_short, tau_long, R2_long, r) for r in rmax_grid]

fig, ax = plt.subplots()
ax.plot(rmax_grid, deltas, color="C0", lw=2)
ax.axhline(1.0, ls="--", color="grey", label=r"$\delta^* = 1$")
ax.axvline(R2_max_default, ls=":", color="C3",
           label=f"Oster default {R2_max_default:0.2f}")
ax.set_xlabel(r"$R^2_{\max}$")
ax.set_ylabel(r"$\delta^*$ (selection ratio that drives $\hat\tau$ to 0)")
ax.set_title("Sensitivity of Oster's $\\delta^*$ to the $R^2_{\\max}$ assumption")
ax.legend()
plt.tight_layout(); plt.close(fig)
np.array(deltas)[[0, 20, 40, 60, 79]]

**Read the curve.** As $R^2_{\max}$ rises toward 1, the denominator in the $\delta^*$ formula
grows, so $\delta^*$ falls. The honest reporting move is to say "our result survives $\delta > 1$ for
any $R^2_{\max} \le X$." If the threshold $X$ is small (e.g., $R^2_{\max} < 0.20$), you do *not*
have a robust selection-on-unobservables defense. If $X$ extends to $1$, you do.

## 5. Monte Carlo: does the recipe recover the truth?

We have one realization. To know whether the procedure works in repeated samples, we re-draw the DGP
$M$ times under two regimes:

- **Regime A: $\lambda_1 = \lambda_2$.** Unobservables and observables have equal selection strength.
  Oster's $\delta^*$ should center near 1 (because unobservable selection is exactly as strong as
  observable selection, so wiping out the residual bias requires $\delta \approx 1$). The bias-
  adjusted $\beta^*$ (at $\delta = 1$) should be near the truth $\tau = 0.3$.
- **Regime B: $\lambda_1 \gg \lambda_2$.** Observables are much more informative than unobservables.
  $\delta^*$ should be **larger** because little selection is left to undo.

In [ ]:
def one_oster_sim(seed, lambda1, lambda2):
    rng_s = np.random.default_rng(seed)
    df_s = make_oster_dgp(rng_s, lambda1=lambda1, lambda2=lambda2)
    tau_s, R2_s, tau_l, R2_l = run_short_long(df_s)
    R2max = min(1.3 * R2_l, 0.99)
    d_star = oster_delta(tau_s, R2_s, tau_l, R2_l, R2max)
    b_star = oster_beta_star(tau_s, R2_s, tau_l, R2_l, R2max, 1.0)
    return d_star, b_star, tau_l

M = 300
recs = {"A (lambda1=lambda2=0.5)": [], "B (lambda1=0.8, lambda2=0.2)": []}
for m in range(M):
    recs["A (lambda1=lambda2=0.5)"].append(one_oster_sim(seed=2000+m, lambda1=0.5, lambda2=0.5))
    recs["B (lambda1=0.8, lambda2=0.2)"].append(one_oster_sim(seed=4000+m, lambda1=0.8, lambda2=0.2))

agg = {}
for name, lst in recs.items():
    arr = np.array(lst)
    agg[name] = {"mean delta*": arr[:,0].mean(),
                 "mean beta* (delta=1)": arr[:,1].mean(),
                 "mean tau_long": arr[:,2].mean()}
pd.DataFrame(agg).T

**Read the simulation table.** In Regime A, $\widehat{\delta^*}$ centers near 1 and the bias-
adjusted $\beta^*$ centers near the truth $0.30$. In Regime B, $\widehat{\delta^*}$ is substantially
larger - the observables soaked up most of the selection, so a much smaller fraction is left to be
explained by the unobservables. The recipe works *in expectation*; it does not work for any single
realization with small $N$, which is why Oster's own practice is to combine the $\delta$ statistic with
a confidence interval (her appendix gives a parametric bootstrap).

## 6. A bootstrap confidence interval for $\delta^*$

A single point estimate of $\delta^*$ is fragile; even small sampling fluctuations in
$(\widehat\tau_{\text{short}}, R^2_{\text{short}}, \widehat\tau_{\text{long}}, R^2_{\text{long}})$
can push $\delta^*$ around by a lot. Oster's appendix suggests a non-parametric bootstrap: draw
$B$ samples with replacement, recompute the four pieces, recompute $\delta^*$, and report the
5th-95th percentile band.

In [ ]:
def bootstrap_delta(df, R2_max, B=500, seed=21):
    rng_b = np.random.default_rng(seed)
    n = len(df)
    deltas = np.zeros(B)
    for b in range(B):
        idx = rng_b.choice(n, size=n, replace=True)
        sub = df.iloc[idx]
        ts, rs, tl, rl = run_short_long(sub)
        if abs(rl - rs) < 1e-6 or abs(R2_max - rl) < 1e-6:
            deltas[b] = np.nan
        else:
            deltas[b] = oster_delta(ts, rs, tl, rl, R2_max)
    deltas = deltas[~np.isnan(deltas)]
    return np.percentile(deltas, [5, 50, 95]), deltas

(q5, q50, q95), bs_deltas = bootstrap_delta(df, R2_max_default, B=400)
print(f"bootstrap median delta* = {q50:0.3f}")
print(f"bootstrap [5%, 95%]     = [{q5:0.3f}, {q95:0.3f}]")
print(f"share of replications with delta* > 1: {(bs_deltas > 1).mean():0.3f}")

**Read the band.** If the bootstrap [5%, 95%] interval for $\delta^*$ excludes 1, the paper
has a defensible robustness claim. If it straddles 1, the headline does not survive plausible
selection on unobservables.

## 7. When Oster $\delta$ fails (be honest)

1. **Highly non-linear selection.** Oster's derivation is for a linear model with constant
   coefficient of proportionality between observed and unobserved selection. If the unobservables
   interact non-linearly with the observables, the formula is misspecified. The diagnostic is to
   re-run the analysis on subsamples and check that $\delta^*$ is stable.

2. **Many weak controls.** If the "long" regression adds many controls that individually contribute
   little to $R^2$, the denominator $R^2_{\text{long}} - R^2_{\text{short}}$ is tiny and $\delta^*$
   becomes wildly unstable. Pre-screen and report only blocks of controls that *jointly* move $R^2$
   by at least a few percentage points.

3. **The $R^2_{\max}$ debate.** There is no consensus default. Always report the curve from section 4
   and let the reader draw the line.

## 8. Your turn

1. **Selection sign flip.** Change `lambda2` to $-0.5$. Now the unobservables push the treatment the
   opposite way from the observables. What happens to $\delta^*$? Does the bias-adjusted $\beta^*$
   over- or under-shoot?
2. **Strong unobservables.** Increase `theta2` to 1.2 and `lambda2` to 0.8 (powerful unobservables
   that strongly select treatment). Does $\widehat\tau_{\text{long}}$ still look credible? What does
   $\delta^*$ tell you?
3. **$R^2_{\max}$ disclosure norm.** Pick three values of $R^2_{\max}$ ($1.3 \cdot \widetilde R^2$,
   $1.5 \cdot \widetilde R^2$, and $1.0$) and report the implied $\delta^*$ in a three-line table that
   you could paste straight into a paper appendix.

**Citations.** Oster (2019, *Journal of Business & Economic Statistics* 37(2):187-204); Altonji, Elder
& Taber (2005, *Journal of Political Economy* 113(1):151-184) for the original selection-on-
unobservables framework.